# Clase 1 — Demo: El Ciclo del Agente Inteligente

**IADS 3 | Desarrollo de Sistemas de IA — Unidad 1**

Todavía no vimos APIs ni modelos reales (eso arranca en la Clase 3), así que en esta demo vamos a **simular en Python puro** el ciclo Percepción → Razonamiento → Acción que estudiamos en la teoría de agentes (Russell & Norvig) y el marco PEAS.

El objetivo es *ver* funcionando lo que hasta ahora fue solo diagrama.

## Ejemplo 1: El Termostato Inteligente

Es el ejemplo más simple de agente: no tiene Base de Conocimiento externa, solo una regla fija.

- **Entorno:** el aire de la habitación (simulado con una variable de temperatura).
- **Sensores:** un termómetro (una función que "lee" la temperatura).
- **Razonamiento:** una regla simple (si temp < 18°C, encender).
- **Actuador:** el interruptor de la caldera (una función que cambia el estado).

In [ ]:
import random

class Termostato:
    def __init__(self):
        self.caldera_encendida = False

    # SENSOR: percibe el entorno
    def percibir(self, temperatura_ambiente):
        print(f"Sensor -> Temperatura detectada: {temperatura_ambiente}°C")
        return temperatura_ambiente

    # RAZONAMIENTO: decide en base a una regla fija
    def razonar(self, temperatura):
        if temperatura < 18:
            return "encender"
        elif temperatura > 24:
            return "apagar"
        else:
            return "mantener"

    # ACTUADOR: ejecuta la acción sobre el entorno
    def actuar(self, decision):
        if decision == "encender" and not self.caldera_encendida:
            self.caldera_encendida = True
            print("Actuador -> Caldera ENCENDIDA")
        elif decision == "apagar" and self.caldera_encendida:
            self.caldera_encendida = False
            print("Actuador -> Caldera APAGADA")
        else:
            print(f"Actuador -> Sin cambios (caldera {'encendida' if self.caldera_encendida else 'apagada'})")

    # CICLO DEL AGENTE
    def ciclo(self, temperatura_ambiente):
        percepcion = self.percibir(temperatura_ambiente)
        decision = self.razonar(percepcion)
        self.actuar(decision)
        print("-" * 40)


# Simulamos el entorno cambiando en el tiempo (ej: baja la temperatura de noche)
agente = Termostato()
temperaturas_del_dia = [22, 19, 17, 15, 16, 20, 25]

for temp in temperaturas_del_dia:
    agente.ciclo(temp)

> **Para discutir en clase:** este agente nunca se equivoca porque su regla es fija y determinista. Pero si el entorno cambia (ej. una casa con muchas ventanas donde 18°C se siente distinto que en otra con buena aislación), el agente no se adapta — necesitaría una Base de Conocimiento para "aprender" el contexto. Ahí es donde entra el 5to pilar del PEAS extendido.

## Ejemplo 2: Mini-agente de Onboarding — Caso Ortelana Textil

Ahora agregamos el **5to pilar**: la Base de Conocimiento. Este agente no tiene una sola regla fija, sino que consulta un "repositorio" de reglas de negocio antes de decidir. Todavía no hay LLM ni API (eso viene en la Clase 3) — la Base de Conocimiento es simplemente un diccionario en memoria, para que el concepto quede claro sin depender de tecnología externa todavía.

- **Entorno:** solicitudes de alta de distribuidores.
- **Sensores:** los datos que llegan en cada solicitud (CUIT, monto solicitado).
- **Base de Conocimiento:** políticas de crédito de Ortelana Textil.
- **Razonamiento:** cruza la solicitud contra las políticas.
- **Actuadores:** aprobar, rechazar o pedir revisión manual.

In [ ]:
class AgenteOnboardingOrtelana:
    def __init__(self):
        # BASE DE CONOCIMIENTO: reglas de negocio de Ortelana Textil
        self.base_de_conocimiento = {
            "limite_alta_automatica": 500_000,   # ARS
            "limite_rechazo_directo": 5_000_000, # ARS
            "cuits_en_lista_negra": ["20111111112"]
        }

    # SENSOR
    def percibir(self, solicitud):
        print(f"Sensor -> Solicitud recibida: {solicitud}")
        return solicitud

    # RAZONAMIENTO: consulta la base de conocimiento antes de decidir
    def razonar(self, solicitud):
        cuit = solicitud["cuit"]
        monto = solicitud["monto_solicitado"]

        if cuit in self.base_de_conocimiento["cuits_en_lista_negra"]:
            return "rechazar"
        if monto <= self.base_de_conocimiento["limite_alta_automatica"]:
            return "aprobar_automatico"
        if monto >= self.base_de_conocimiento["limite_rechazo_directo"]:
            return "rechazar"
        return "revision_manual"

    # ACTUADOR
    def actuar(self, decision, solicitud):
        acciones = {
            "aprobar_automatico": f"Cuenta creada automáticamente para CUIT {solicitud['cuit']}",
            "rechazar": f"Solicitud de CUIT {solicitud['cuit']} RECHAZADA",
            "revision_manual": f"CUIT {solicitud['cuit']} enviado a revisión manual (monto alto)"
        }
        print(f"Actuador -> {acciones[decision]}")

    def ciclo(self, solicitud):
        percepcion = self.percibir(solicitud)
        decision = self.razonar(percepcion)
        self.actuar(decision, percepcion)
        print("-" * 40)


agente = AgenteOnboardingOrtelana()

solicitudes = [
    {"cuit": "20304567899", "monto_solicitado": 300_000},
    {"cuit": "20111111112", "monto_solicitado": 100_000},
    {"cuit": "20999999998", "monto_solicitado": 1_500_000},
    {"cuit": "20888888887", "monto_solicitado": 8_000_000},
]

for s in solicitudes:
    agente.ciclo(s)

### Preguntas para cerrar la práctica con los alumnos

1. En el Ejemplo 2, ¿cuál sería el **Objetivo (Performance)** de este agente? (pista: reducir tiempo de alta manteniendo bajo riesgo de crédito)
2. Si mañana Ortelana cambia su política de crédito, ¿hay que tocar el código del agente o solo la Base de Conocimiento? ¿Por qué esto es importante para un sistema en producción?
3. ¿Qué pasaría si en vez de un diccionario, la Base de Conocimiento fuera una base de datos SQL real o una base vectorial? ¿Cambia el razonamiento del agente o solo *de dónde* saca la información?

> **Spoiler de lo que viene:** en la Clase 3 vamos a reemplazar la percepción "prolija" de estos ejemplos (un diccionario con datos ya limpios) por texto libre y desestructurado real, y vamos a necesitar un LLM (vía API) que actúe como "normalizador semántico" antes de que este mismo razonamiento determinista pueda aplicarse.